In [ ]:
import os
os.chdir("..")

from data.data import get_lp_dataloaders
from utils.utils import set_seed
from scripts.lp_script import construct_backbone, train_classifier, test
import torch
from models.models import Classifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 1. Floor: Randomly initialize a backbone and train a classifier on top of it
SEED = 2952
set_seed(SEED, deterministic=True, benchmark=False) # benchmark=False for reproducibility

######## HYPERPARAMETERS ########
batch_size = 256
workers = 32
epochs=100
lr=3e-3
wd=0.05
out_dir="results/baselines/random_init"
arch = 'vit_s'
use_amp=True
label_smoothing=0.05

######## PREPARE DATA, MODEL ########
train_loader, val_loader, test_loader = get_lp_dataloaders(batch_size=batch_size, workers=workers, normalization="IN")
backbone = construct_backbone(arch=arch, device=device)
model = Classifier(backbone=backbone, num_classes=10, requires_grad=False, eval_mode=True)

######## TRAINING ########
train_classifier(epochs=epochs,
                 model=model,
                 train_loader=train_loader,
                 val_loader=val_loader,
                 lr=lr,
                 wd=wd,
                 device=device,
                 out_dir=out_dir,
                 use_amp=use_amp,
                 label_smoothing=label_smoothing,
                 print_freq=5)

######## TESTING ########
best_model_path = f"{out_dir}/best.ckpt"
test(model, test_loader, device, ckpt_path=best_model_path, out_dir=out_dir)

No checkpoint path provided, returning randomly initialized model.
Epoch 001/100 | lr 3.00e-03 | train_loss 2.1740 | val_loss 2.0687 | val_acc 26.78% | 25.5s/6.5s (train/val)
Epoch 005/100 | lr 2.98e-03 | train_loss 1.9591 | val_loss 1.9515 | val_acc 31.85% | 11.4s/6.8s (train/val)


Exception in thread Thread-15 (_pin_memory_loop):
Traceback (most recent call last):
  File "/home/liue/miniconda3/envs/trident/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/home/liue/.local/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "/home/liue/miniconda3/envs/trident/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/home/liue/.local/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py", line 61, in _pin_memory_loop
    do_one_step()
  File "/home/liue/.local/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py", line 37, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
  File "/home/liue/miniconda3/envs/trident/lib/python3.10/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
  File "/home/liue/.local/lib/python3.10/site-packages/torch/multip

KeyboardInterrupt: 

In [3]:
# 2. Supervised: Make backbone trainable and train the classifier End-to-End
######## HYPERPARAMETERS ########
batch_size = 256
workers = 32
epochs=100
lr=3e-3
wd=0.05
backbone_lr=3e-4
backbone_wd=0.05
out_dir="results/baselines/e2e_training"
arch = 'vit_s'
use_amp=True
label_smoothing=0.05

new_backbone = construct_backbone(arch=arch, device=device)
model = Classifier(backbone=new_backbone, num_classes=10, requires_grad=True, eval_mode=False)

######## TRAINING ########
train_classifier(epochs=epochs,
                 model=model,
                 train_loader=train_loader,
                 val_loader=val_loader,
                 lr=lr,
                 wd=wd,
                 device=device,
                 out_dir=out_dir,
                 use_amp=use_amp,
                 backbone_lr=backbone_lr,
                 backbone_wd=backbone_wd,
                 label_smoothing=label_smoothing,
                 print_freq=5)

######## TESTING ########
best_model_path = f"{out_dir}/best.ckpt"
test(model, test_loader, device, ckpt_path=best_model_path, out_dir=out_dir)

No checkpoint path provided, returning randomly initialized model.
Epoch 001/100 | lr 3.00e-04 | train_loss 2.1491 | val_loss 1.8594 | val_acc 32.54% | 19.9s/7.2s (train/val)
Epoch 005/100 | lr 2.98e-04 | train_loss 1.1949 | val_loss 1.1637 | val_acc 65.77% | 19.0s/7.2s (train/val)
Epoch 010/100 | lr 2.93e-04 | train_loss 0.9149 | val_loss 1.0566 | val_acc 69.56% | 18.1s/7.2s (train/val)
Epoch 015/100 | lr 2.84e-04 | train_loss 0.7937 | val_loss 1.0389 | val_acc 71.12% | 18.7s/7.0s (train/val)
Epoch 020/100 | lr 2.71e-04 | train_loss 0.6591 | val_loss 1.0948 | val_acc 71.72% | 18.6s/7.2s (train/val)


KeyboardInterrupt: 

In [ ]:
# 3. Pretrained Using ImageNet ViT weights + Linear Probing
######## HYPERPARAMETERS ########
batch_size = 256
workers = 32
epochs=100
lr=3e-3
wd=0.05
out_dir="results/baselines/imagenet_pretrained"
arch = "vit_small_patch16_224"   # ViT-Small, ImageNet-1k pretrained
use_amp=True
label_smoothing=0.05

######## PREPARE DATA ########
train_loader, val_loader, test_loader = get_lp_dataloaders(batch_size=batch_size, workers=workers)

######## MODEL ########
from timm import create_model
# Load ViT backbone with ImageNet pretrained weights
backbone = create_model(arch, pretrained=True, num_classes=0)  # num_classes=0 removes head
backbone.reset_classifier(num_classes=0, global_pool="token")

# Freeze backbone (linear probe)
for p in backbone.parameters():
    p.requires_grad = False

# Add linear classifier
model = Classifier(backbone=backbone, num_classes=10,
                   requires_grad=False, eval_mode=True)

######## TRAINING ########
train_classifier(epochs=epochs,
                 model=model,
                 train_loader=train_loader,
                 val_loader=val_loader,
                 lr=lr,
                 wd=wd,
                 device=device,
                 out_dir=out_dir,
                 use_amp=use_amp,
                 label_smoothing=label_smoothing,
                 print_freq=5)

######## TESTING ########
best_model_path = f"{out_dir}/best.ckpt"
test(model, test_loader, device, ckpt_path=best_model_path, out_dir=out_dir)

Epoch 001/100 | lr 3.00e-03 | train_loss 1.8871 | val_loss 1.6363 | val_acc 42.97% | 12.2s/6.7s (train/val)
Epoch 005/100 | lr 2.98e-03 | train_loss 1.4225 | val_loss 1.4414 | val_acc 52.11% | 11.3s/7.2s (train/val)
Epoch 010/100 | lr 2.93e-03 | train_loss 1.3710 | val_loss 1.4591 | val_acc 50.52% | 11.8s/7.3s (train/val)
Epoch 015/100 | lr 2.84e-03 | train_loss 1.3269 | val_loss 1.4217 | val_acc 53.52% | 12.0s/6.7s (train/val)
Epoch 020/100 | lr 2.71e-03 | train_loss 1.3208 | val_loss 1.3818 | val_acc 54.43% | 12.4s/7.4s (train/val)
Epoch 025/100 | lr 2.56e-03 | train_loss 1.3025 | val_loss 1.3601 | val_acc 56.09% | 11.9s/7.1s (train/val)
Epoch 030/100 | lr 2.38e-03 | train_loss 1.2989 | val_loss 1.3661 | val_acc 56.66% | 12.1s/7.1s (train/val)
Epoch 035/100 | lr 2.18e-03 | train_loss 1.2839 | val_loss 1.3676 | val_acc 55.59% | 12.5s/7.3s (train/val)
Epoch 040/100 | lr 1.96e-03 | train_loss 1.2663 | val_loss 1.3706 | val_acc 55.50% | 11.3s/7.1s (train/val)
Epoch 045/100 | lr 1.74e-03 

(1.1590057809027636, 0.5834272862018297)